In [0]:
%sql
-- 1. Customers Table
CREATE TABLE customers (
    customer_id INT,
    customer_name STRING,
    customer_city STRING
);

-- 2. Cars Table
CREATE TABLE cars (
    car_id INT,
    brand STRING,
    model STRING,
    price DOUBLE
);

-- 3. Sales Table
CREATE TABLE sales (
    sale_id INT,
    customer_id INT,
    car_id INT,
    sale_date DATE,
    quantity INT
);

-- 4. Dealers Table
CREATE TABLE dealers (
    dealer_id INT,
    dealer_name STRING,
    dealer_city STRING
);

-- 5. Sales Dealer Table
CREATE TABLE sales_dealer (
    sale_id INT,
    dealer_id INT
);

In [0]:
%sql
INSERT INTO customers VALUES
(1, 'Yashwanth', 'Chennai'),
(2, 'Rahul', 'Bangalore'),
(3, 'Sneha', 'Hyderabad'),
(4, 'Amit', 'Mumbai'),
(5, 'Priya', 'Delhi'),
(6, 'Kiran', 'Chennai'),
(7, 'Meena', 'Pune'),
(8, 'Arjun', 'Kolkata'),
(9, 'Divya', 'Chennai'),
(10, 'Ravi', 'Bangalore');

INSERT INTO cars VALUES
(101, 'Toyota', 'Innova', 20000),
(102, 'Honda', 'City', 15000),
(103, 'Hyundai', 'Creta', 18000),
(104, 'Maruti', 'Swift', 8000),
(105, 'Tata', 'Nexon', 12000),
(106, 'BMW', 'X1', 35000),
(107, 'Audi', 'A4', 40000),
(108, 'Kia', 'Seltos', 17000),
(109, 'Mahindra', 'XUV700', 22000),
(110, 'Skoda', 'Octavia', 25000);

INSERT INTO sales VALUES
(1001, 1, 101, '2024-01-10', 1),
(1002, 2, 102, '2024-01-12', 2),
(1003, 3, 103, '2024-01-15', 1),
(1004, 4, 104, '2024-01-18', 3),
(1005, 5, 105, '2024-01-20', 1),
(1006, 6, 106, '2024-01-25', 1),
(1007, 7, 107, '2024-02-01', 2),
(1008, 8, 108, '2024-02-05', 1),
(1009, 9, 109, '2024-02-10', 1),
(1010, 10, 110, '2024-02-15', 2);

INSERT INTO dealers VALUES
(201, 'Dealer One', 'Chennai'),
(202, 'Dealer Two', 'Bangalore'),
(203, 'Dealer Three', 'Mumbai'),
(204, 'Dealer Four', 'Delhi'),
(205, 'Dealer Five', 'Hyderabad'),
(206, 'Dealer Six', 'Pune'),
(207, 'Dealer Seven', 'Kolkata'),
(208, 'Dealer Eight', 'Chennai');

INSERT INTO sales_dealer VALUES
(1001, 201),
(1002, 202),
(1003, 205),
(1004, 203),
(1005, 204),
(1006, 201),
(1007, 206),
(1008, 207),
(1009, 208),
(1010, 202);

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
INSERT INTO customers VALUES
(11, NULL, 'Chennai'),          -- Missing name
(12, 'Anita', NULL),            -- Missing city
(13, 'Rahul', 'Bangalore'),     -- Duplicate name
(14, '', 'Mumbai'),             -- Empty name
(15, 'Vikram', '');             -- Empty city

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
INSERT INTO cars VALUES
(111, NULL, 'ModelX', 30000),   -- Missing brand
(112, 'Tesla', NULL, 50000),    -- Missing model
(113, 'Ford', 'EcoSport', NULL),-- Missing price
(114, '', 'i20', 12000),        -- Empty brand
(115, 'Nissan', '', 14000);     -- Empty model

INSERT INTO sales VALUES
(1011, NULL, 101, '2024-03-01', 1),  -- Missing customer_id
(1012, 1, NULL, '2024-03-05', 1),    -- Missing car_id
(1013, 99, 101, '2024-03-10', 1),    -- Non-existing customer_id
(1014, 1, 999, '2024-03-12', 1),     -- Non-existing car_id
(1015, 2, 102, NULL, 1);             -- Missing date

INSERT INTO dealers VALUES
(209, NULL, 'Chennai'),         -- Missing dealer name
(210, 'Dealer Nine', NULL),     -- Missing city
(211, 'Dealer One', 'Chennai'), -- Duplicate dealer
(212, '', 'Delhi'),             -- Empty name
(213, 'Dealer Ten', '');        -- Empty city

INSERT INTO sales_dealer VALUES
(NULL, 201),    -- Missing sale_id
(1011, NULL),   -- Missing dealer_id
(9999, 201),    -- Non-existing sale_id
(1001, 999),    -- Non-existing dealer_id
(1012, 202);    -- Valid-ish but linked to NULL car sale

num_affected_rows,num_inserted_rows
5,5


## Load data

In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, sum as _sum, count, avg,
    dense_rank, month, year, trim, to_date
)
from pyspark.sql import Window
 
spark = SparkSession.builder.appName("Car-Sales-Pipeline").getOrCreate()

In [0]:
# Load all 5 tables from the Hive metastore into PySpark DataFrames
customers    = spark.sql("SELECT * FROM customers")
cars         = spark.sql("SELECT * FROM cars")
sales        = spark.sql("SELECT * FROM sales")
dealers      = spark.sql("SELECT * FROM dealers")
sales_dealer = spark.sql("SELECT * FROM sales_dealer")

In [0]:
# Print schemas to understand data types
print("=== SCHEMAS ===")
for name, df in [("customers", customers), ("cars", cars), ("sales", sales),
                  ("dealers", dealers), ("sales_dealer", sales_dealer)]:
    print(f"\n--- {name} ---")
    df.printSchema()

=== SCHEMAS ===

--- customers ---
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_city: string (nullable = true)


--- cars ---
root
 |-- car_id: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: double (nullable = true)


--- sales ---
root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- car_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)


--- dealers ---
root
 |-- dealer_id: integer (nullable = true)
 |-- dealer_name: string (nullable = true)
 |-- dealer_city: string (nullable = true)


--- sales_dealer ---
root
 |-- sale_id: integer (nullable = true)
 |-- dealer_id: integer (nullable = true)



In [0]:
# Row counts for each table
print("=== ROW COUNTS (raw) ===")
for name, df in [("customers", customers), ("cars", cars), ("sales", sales),
                  ("dealers", dealers), ("sales_dealer", sales_dealer)]:
    print(f"{name}: {df.count()} rows")

=== ROW COUNTS (raw) ===
customers: 15 rows
cars: 15 rows
sales: 15 rows
dealers: 13 rows
sales_dealer: 15 rows


In [0]:
# Identify NULLs and empty strings in each table
print("\n=== NULL / EMPTY CHECKS ===")
 
# customers
print("\n[customers] NULLs:")
customers.filter(
    col("customer_id").isNull() |
    col("customer_name").isNull() | (col("customer_name") == "") |
    col("customer_city").isNull() | (col("customer_city") == "")
).show()
 
# cars
print("[cars] NULLs / negative price:")
cars.filter(
    col("car_id").isNull() |
    col("brand").isNull() | (col("brand") == "") |
    col("model").isNull() | (col("model") == "") |
    col("price").isNull() | (col("price") < 0)
).show()
 
# sales
print("[sales] NULLs:")
sales.filter(
    col("sale_id").isNull() |
    col("customer_id").isNull() |
    col("car_id").isNull() |
    col("sale_date").isNull()
).show()
 
# dealers
print("[dealers] NULLs / empty:")
dealers.filter(
    col("dealer_id").isNull() |
    col("dealer_name").isNull() | (col("dealer_name") == "") |
    col("dealer_city").isNull() | (col("dealer_city") == "")
).show()
 
# sales_dealer
print("[sales_dealer] NULLs:")
sales_dealer.filter(
    col("sale_id").isNull() | col("dealer_id").isNull()
).show()


=== NULL / EMPTY CHECKS ===

[customers] NULLs:
+-----------+-------------+-------------+
|customer_id|customer_name|customer_city|
+-----------+-------------+-------------+
|         11|         NULL|      Chennai|
|         12|        Anita|         NULL|
|         14|             |       Mumbai|
|         15|       Vikram|             |
+-----------+-------------+-------------+

[cars] NULLs / negative price:
+------+------+--------+-------+
|car_id| brand|   model|  price|
+------+------+--------+-------+
|   111|  NULL|  ModelX|30000.0|
|   112| Tesla|    NULL|50000.0|
|   113|  Ford|EcoSport|   NULL|
|   114|      |     i20|12000.0|
|   115|Nissan|        |14000.0|
+------+------+--------+-------+

[sales] NULLs:
+-------+-----------+------+----------+--------+
|sale_id|customer_id|car_id| sale_date|quantity|
+-------+-----------+------+----------+--------+
|   1011|       NULL|   101|2024-03-01|       1|
|   1012|          1|  NULL|2024-03-05|       1|
|   1015|          2|   1

# PHASE 2 – CLEANING
# Each table is cleaned independently → produces a clean DataFrame.

In [0]:
# Clean customers
# drop rows with NULL customer_id (can't identify the record).
#  Fill NULL/empty name  → "Unknown"
#  Fill NULL/empty city  → "Unknown"
#  Trim whitespace from string fields.
customers_clean = (
    customers
    .filter(col("customer_id").isNotNull())          # drop un-identifiable rows
    .withColumn("customer_name",
        when(col("customer_name").isNull() | (trim(col("customer_name")) == ""),
             lit("Unknown"))
        .otherwise(trim(col("customer_name"))))
    .withColumn("customer_city",
        when(col("customer_city").isNull() | (trim(col("customer_city")) == ""),
             lit("Unknown"))
        .otherwise(trim(col("customer_city"))))
)
print("customers_clean count:", customers_clean.count())
customers_clean.show()

customers_clean count: 15
+-----------+-------------+-------------+
|customer_id|customer_name|customer_city|
+-----------+-------------+-------------+
|          1|    Yashwanth|      Chennai|
|          2|        Rahul|    Bangalore|
|          3|        Sneha|    Hyderabad|
|          4|         Amit|       Mumbai|
|          5|        Priya|        Delhi|
|          6|        Kiran|      Chennai|
|          7|        Meena|         Pune|
|          8|        Arjun|      Kolkata|
|          9|        Divya|      Chennai|
|         10|         Ravi|    Bangalore|
|         11|      Unknown|      Chennai|
|         12|        Anita|      Unknown|
|         13|        Rahul|    Bangalore|
|         14|      Unknown|       Mumbai|
|         15|       Vikram|      Unknown|
+-----------+-------------+-------------+



In [0]:
# Clean cars
# drop rows with NULL car_id (car_id is the primary key, a record can't exists without it).
# Fill NULL/empty brand → "Unknown"
# Fill NULL/empty model → "Unknown"
# Fill NULL price       → 0.0  (price=0 flags the record; avoids NaN in revenue)
# Fix negative price    → abs(price) 
cars_clean = (
    cars
    .filter(col("car_id").isNotNull())               # primary key must exist
    .withColumn("brand",
        when(col("brand").isNull() | (trim(col("brand")) == ""),
             lit("Unknown"))
        .otherwise(trim(col("brand"))))
    .withColumn("model",
        when(col("model").isNull() | (trim(col("model")) == ""),
             lit("Unknown"))
        .otherwise(trim(col("model"))))
    .withColumn("price",
        when(col("price").isNull(), lit(0.0))        # NULL → 0
        .when(col("price") < 0, -col("price"))       # negative → positive
        .otherwise(col("price")))
)
print("cars_clean count:", cars_clean.count())
cars_clean.show()

cars_clean count: 15
+------+--------+--------+-------+
|car_id|   brand|   model|  price|
+------+--------+--------+-------+
|   101|  Toyota|  Innova|20000.0|
|   102|   Honda|    City|15000.0|
|   103| Hyundai|   Creta|18000.0|
|   104|  Maruti|   Swift| 8000.0|
|   105|    Tata|   Nexon|12000.0|
|   106|     BMW|      X1|35000.0|
|   107|    Audi|      A4|40000.0|
|   108|     Kia|  Seltos|17000.0|
|   109|Mahindra|  XUV700|22000.0|
|   110|   Skoda| Octavia|25000.0|
|   111| Unknown|  ModelX|30000.0|
|   112|   Tesla| Unknown|50000.0|
|   113|    Ford|EcoSport|    0.0|
|   114| Unknown|     i20|12000.0|
|   115|  Nissan| Unknown|14000.0|
+------+--------+--------+-------+



In [0]:
# Clean sales
# drop rows where customer_id OR car_id is NULL (FK columns — unusable without them).
# Drop rows where sale_id is NULL (primary key).
# Fill NULL sale_date → "1970-01-01" (sentinel date, easy to filter later).

sales_clean = (
    sales
    .filter(
        col("sale_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("car_id").isNotNull()
    )
    .withColumn("sale_date",
        when(col("sale_date").isNull(), to_date(lit("1970-01-01")))
        .otherwise(col("sale_date")))
)
print("sales_clean count:", sales_clean.count())
sales_clean.show()

sales_clean count: 13
+-------+-----------+------+----------+--------+
|sale_id|customer_id|car_id| sale_date|quantity|
+-------+-----------+------+----------+--------+
|   1001|          1|   101|2024-01-10|       1|
|   1002|          2|   102|2024-01-12|       2|
|   1003|          3|   103|2024-01-15|       1|
|   1004|          4|   104|2024-01-18|       3|
|   1005|          5|   105|2024-01-20|       1|
|   1006|          6|   106|2024-01-25|       1|
|   1007|          7|   107|2024-02-01|       2|
|   1008|          8|   108|2024-02-05|       1|
|   1009|          9|   109|2024-02-10|       1|
|   1010|         10|   110|2024-02-15|       2|
|   1013|         99|   101|2024-03-10|       1|
|   1014|          1|   999|2024-03-12|       1|
|   1015|          2|   102|1970-01-01|       1|
+-------+-----------+------+----------+--------+



In [0]:
# Clean dealers
# drop rows with NULL dealer_id.
# Fill NULL/empty dealer_name → "Unknown Dealer"
# Fill NULL/empty dealer_city → "Unknown"
dealers_clean = (
    dealers
    .filter(col("dealer_id").isNotNull())
    .withColumn("dealer_name",
        when(col("dealer_name").isNull() | (trim(col("dealer_name")) == ""),
             lit("Unknown Dealer"))
        .otherwise(trim(col("dealer_name"))))
    .withColumn("dealer_city",
        when(col("dealer_city").isNull() | (trim(col("dealer_city")) == ""),
             lit("Unknown"))
        .otherwise(trim(col("dealer_city"))))
)
print("dealers_clean count:", dealers_clean.count())
dealers_clean.show()

dealers_clean count: 13
+---------+--------------+-----------+
|dealer_id|   dealer_name|dealer_city|
+---------+--------------+-----------+
|      201|    Dealer One|    Chennai|
|      202|    Dealer Two|  Bangalore|
|      203|  Dealer Three|     Mumbai|
|      204|   Dealer Four|      Delhi|
|      205|   Dealer Five|  Hyderabad|
|      206|    Dealer Six|       Pune|
|      207|  Dealer Seven|    Kolkata|
|      208|  Dealer Eight|    Chennai|
|      209|Unknown Dealer|    Chennai|
|      210|   Dealer Nine|    Unknown|
|      211|    Dealer One|    Chennai|
|      212|Unknown Dealer|      Delhi|
|      213|    Dealer Ten|    Unknown|
+---------+--------------+-----------+



In [0]:
# Clean sales_dealer
# drop any row where sale_id OR dealer_id is NULL (both are FKs, both required).
sales_dealer_clean = (
    sales_dealer
    .filter(col("sale_id").isNotNull() & col("dealer_id").isNotNull())
)
print("sales_dealer_clean count:", sales_dealer_clean.count())
sales_dealer_clean.show()

sales_dealer_clean count: 13
+-------+---------+
|sale_id|dealer_id|
+-------+---------+
|   1001|      201|
|   1002|      202|
|   1003|      205|
|   1004|      203|
|   1005|      204|
|   1006|      201|
|   1007|      206|
|   1008|      207|
|   1009|      208|
|   1010|      202|
|   9999|      201|
|   1001|      999|
|   1012|      202|
+-------+---------+



# PHASE 3 – VALIDATION  (left_anti joins = find orphan FK rows)

In [0]:
# Find sales referencing non-existent customers (before FK fix)
invalid_sales_customer = (
    sales_clean
    .join(customers_clean.select("customer_id"), "customer_id", "left_anti")
)
print("Sales with invalid customer_id:", invalid_sales_customer.count())
invalid_sales_customer.show()

Sales with invalid customer_id: 1
+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|   1013|   101|2024-03-10|       1|
+-----------+-------+------+----------+--------+



In [0]:
# Find sales referencing non-existent cars (before FK fix)
invalid_sales_car = (
    sales_clean
    .join(cars_clean.select("car_id"), "car_id", "left_anti")
)
print("Sales with invalid car_id:", invalid_sales_car.count())
invalid_sales_car.show()

Sales with invalid car_id: 1
+------+-------+-----------+----------+--------+
|car_id|sale_id|customer_id| sale_date|quantity|
+------+-------+-----------+----------+--------+
|   999|   1014|          1|2024-03-12|       1|
+------+-------+-----------+----------+--------+



In [0]:
# Fix sales_clean: remove rows with invalid FK references.
# This keeps sales_clean as a pure sales table (no extra columns from joins).
# Strategy: inner join on FK columns only to keep valid sale rows.
valid_customer_ids = customers_clean.select("customer_id")
valid_car_ids      = cars_clean.select("car_id")
valid_dealer_ids   = dealers_clean.select("dealer_id")
valid_sale_ids     = sales_clean.select("sale_id")  # will refine after FK check
 
sales_clean = (
    sales_clean
    .join(valid_customer_ids, "customer_id", "inner")  # remove orphan customer refs
    .join(valid_car_ids,      "car_id",      "inner")  # remove orphan car refs
)
print("sales_clean after FK validation:", sales_clean.count())
sales_clean.show()

sales_clean after FK validation: 11
+------+-----------+-------+----------+--------+
|car_id|customer_id|sale_id| sale_date|quantity|
+------+-----------+-------+----------+--------+
|   101|          1|   1001|2024-01-10|       1|
|   102|          2|   1002|2024-01-12|       2|
|   103|          3|   1003|2024-01-15|       1|
|   104|          4|   1004|2024-01-18|       3|
|   105|          5|   1005|2024-01-20|       1|
|   106|          6|   1006|2024-01-25|       1|
|   107|          7|   1007|2024-02-01|       2|
|   108|          8|   1008|2024-02-05|       1|
|   109|          9|   1009|2024-02-10|       1|
|   110|         10|   1010|2024-02-15|       2|
|   102|          2|   1015|1970-01-01|       1|
+------+-----------+-------+----------+--------+



In [0]:
# Fix sales_dealer_clean: remove rows pointing to invalid sales/dealers.
sales_dealer_clean = (
    sales_dealer_clean
    .join(sales_clean.select("sale_id"),"sale_id","inner")
    .join(valid_dealer_ids,"dealer_id", "inner")
)
print("sales_dealer_clean after FK validation:", sales_dealer_clean.count())
sales_dealer_clean.show()

sales_dealer_clean after FK validation: 10
+---------+-------+
|dealer_id|sale_id|
+---------+-------+
|      201|   1001|
|      202|   1002|
|      205|   1003|
|      203|   1004|
|      204|   1005|
|      201|   1006|
|      206|   1007|
|      207|   1008|
|      208|   1009|
|      202|   1010|
+---------+-------+



In [0]:
# Validation report summary
print("\n========== VALIDATION REPORT ==========")
print(f"customers_clean  : {customers_clean.count()} valid rows")
print(f"cars_clean       : {cars_clean.count()} valid rows")
print(f"sales_clean      : {sales_clean.count()} valid rows")
print(f"dealers_clean    : {dealers_clean.count()} valid rows")
print(f"sales_dealer_clean: {sales_dealer_clean.count()} valid rows")
 
invalid_after = (
    sales_clean
    .join(customers_clean.select("customer_id"), "customer_id", "left_anti")
)
print(f"Orphan sales (customer FK): {invalid_after.count()}  ← should be 0")
 
invalid_after_car = (
    sales_clean
    .join(cars_clean.select("car_id"), "car_id", "left_anti")
)
print(f"Orphan sales (car FK) : {invalid_after_car.count()}  ← should be 0")
print("========================================\n")


========== VALIDATION REPORT ==========
customers_clean  : 15 valid rows
cars_clean       : 15 valid rows
sales_clean      : 11 valid rows
dealers_clean    : 13 valid rows
sales_dealer_clean: 10 valid rows
Orphan sales (customer FK): 0  ← should be 0
Orphan sales (car FK) : 0  ← should be 0



# PHASE 4 – TRANSFORMATION
# Build a clean master DataFrame (full_df) by joining the validated tables.
# All joins use explicit column selects → NO redundant/ambiguous columns.

In [0]:
# Build full master DataFrame
# sales_clean already only has: sale_id, customer_id, car_id, sale_date, quantity
# We enrich it by joining dimension tables and selecting only the columns we need.
full_df = (
    sales_clean # fact table (core)
    .join(
        customers_clean.select("customer_id", "customer_name", "customer_city"),
        "customer_id", "inner"
    )
    .join(
        cars_clean.select("car_id", "brand", "model", "price"),
        "car_id", "inner"
    )
    .join(
        sales_dealer_clean.select("sale_id", "dealer_id"),
        "sale_id", "inner"
    )
    .join(
        dealers_clean.select("dealer_id", "dealer_name", "dealer_city"),
        "dealer_id", "inner"
    )
    # Add a revenue column = price × quantity (no ambiguity now — one 'price')
    .withColumn("revenue", col("price") * col("quantity"))
)
 
# Verify no duplicate column names
print("full_df columns:", full_df.columns)
print("full_df count:", full_df.count())
full_df.show()

full_df columns: ['dealer_id', 'sale_id', 'car_id', 'customer_id', 'sale_date', 'quantity', 'customer_name', 'customer_city', 'brand', 'model', 'price', 'dealer_name', 'dealer_city', 'revenue']
full_df count: 10
+---------+-------+------+-----------+----------+--------+-------------+-------------+--------+-------+-------+------------+-----------+-------+
|dealer_id|sale_id|car_id|customer_id| sale_date|quantity|customer_name|customer_city|   brand|  model|  price| dealer_name|dealer_city|revenue|
+---------+-------+------+-----------+----------+--------+-------------+-------------+--------+-------+-------+------------+-----------+-------+
|      201|   1006|   106|          6|2024-01-25|       1|        Kiran|      Chennai|     BMW|     X1|35000.0|  Dealer One|    Chennai|35000.0|
|      202|   1010|   110|         10|2024-02-15|       2|         Ravi|    Bangalore|   Skoda|Octavia|25000.0|  Dealer Two|  Bangalore|50000.0|
|      203|   1004|   104|          4|2024-01-18|       3|     

In [0]:
# 4a. Revenue per customer
revenue_per_customer = (
    full_df
    .groupBy("customer_id", "customer_name")
    .agg(_sum("revenue").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
print("=== Revenue per Customer ===")
revenue_per_customer.show()

=== Revenue per Customer ===
+-----------+-------------+-------------+
|customer_id|customer_name|total_revenue|
+-----------+-------------+-------------+
|          7|        Meena|      80000.0|
|         10|         Ravi|      50000.0|
|          6|        Kiran|      35000.0|
|          2|        Rahul|      30000.0|
|          4|         Amit|      24000.0|
|          9|        Divya|      22000.0|
|          1|    Yashwanth|      20000.0|
|          3|        Sneha|      18000.0|
|          8|        Arjun|      17000.0|
|          5|        Priya|      12000.0|
+-----------+-------------+-------------+



In [0]:
# 4b. Brand-wise sales (total units sold per brand)
brand_wise_sales = (
    full_df
    .groupBy("brand")
    .agg(
        _sum("quantity").alias("total_units_sold"),
        _sum("revenue").alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)
print("=== Brand-wise Sales ===")
brand_wise_sales.show()

=== Brand-wise Sales ===
+--------+----------------+-------------+
|   brand|total_units_sold|total_revenue|
+--------+----------------+-------------+
|    Audi|               2|      80000.0|
|   Skoda|               2|      50000.0|
|     BMW|               1|      35000.0|
|   Honda|               2|      30000.0|
|  Maruti|               3|      24000.0|
|Mahindra|               1|      22000.0|
|  Toyota|               1|      20000.0|
| Hyundai|               1|      18000.0|
|     Kia|               1|      17000.0|
|    Tata|               1|      12000.0|
+--------+----------------+-------------+



In [0]:
# 4c. City-wise revenue (using customer city)
city_revenue = (
    full_df
    .groupBy("customer_city")
    .agg(_sum("revenue").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
print("=== City-wise Revenue ===")
city_revenue.show()

=== City-wise Revenue ===
+-------------+-------------+
|customer_city|total_revenue|
+-------------+-------------+
|    Bangalore|      80000.0|
|         Pune|      80000.0|
|      Chennai|      77000.0|
|       Mumbai|      24000.0|
|    Hyderabad|      18000.0|
|      Kolkata|      17000.0|
|        Delhi|      12000.0|
+-------------+-------------+



# PHASE 5 – DEALER ANALYTICS

In [0]:
# 5a. Revenue per dealer
dealer_revenue = (
    full_df
    .groupBy("dealer_id", "dealer_name", "dealer_city")
    .agg(_sum("revenue").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
print("=== Revenue per Dealer ===")
dealer_revenue.show()

=== Revenue per Dealer ===
+---------+------------+-----------+-------------+
|dealer_id| dealer_name|dealer_city|total_revenue|
+---------+------------+-----------+-------------+
|      206|  Dealer Six|       Pune|      80000.0|
|      202|  Dealer Two|  Bangalore|      80000.0|
|      201|  Dealer One|    Chennai|      55000.0|
|      203|Dealer Three|     Mumbai|      24000.0|
|      208|Dealer Eight|    Chennai|      22000.0|
|      205| Dealer Five|  Hyderabad|      18000.0|
|      207|Dealer Seven|    Kolkata|      17000.0|
|      204| Dealer Four|      Delhi|      12000.0|
+---------+------------+-----------+-------------+



In [0]:
# 5b. Top dealers by revenue (top 3)
print("=== Top 3 Dealers by Revenue ===")
dealer_revenue.limit(3).show()

=== Top 3 Dealers by Revenue ===
+---------+-----------+-----------+-------------+
|dealer_id|dealer_name|dealer_city|total_revenue|
+---------+-----------+-----------+-------------+
|      206| Dealer Six|       Pune|      80000.0|
|      202| Dealer Two|  Bangalore|      80000.0|
|      201| Dealer One|    Chennai|      55000.0|
+---------+-----------+-----------+-------------+



In [0]:
# 5c. Dealer-city contribution
# → What % of total revenue comes from each dealer city?
total_rev = full_df.agg(_sum("revenue").alias("grand_total")).collect()[0]["grand_total"]
 
dealer_city_contribution = (
    full_df
    .groupBy("dealer_city")
    .agg(_sum("revenue").alias("city_revenue"))
    .withColumn("pct_contribution",
        (col("city_revenue") / lit(total_rev) * 100).cast("decimal(5,2)"))
    .orderBy(col("city_revenue").desc())
)
print("=== Dealer-City Revenue Contribution ===")
dealer_city_contribution.show()

=== Dealer-City Revenue Contribution ===
+-----------+------------+----------------+
|dealer_city|city_revenue|pct_contribution|
+-----------+------------+----------------+
|  Bangalore|     80000.0|           25.97|
|       Pune|     80000.0|           25.97|
|    Chennai|     77000.0|           25.00|
|     Mumbai|     24000.0|            7.79|
|  Hyderabad|     18000.0|            5.84|
|    Kolkata|     17000.0|            5.52|
|      Delhi|     12000.0|            3.90|
+-----------+------------+----------------+



# PHASE 6 – SQL ANALYSIS
# Register full_df as a temp view and use Spark SQL for analysis.

In [0]:
# Register temp view for SQL queries
full_df.createOrReplaceTempView("car_sales")

In [0]:
# 6a. Top 3 customers per city (by total revenue, using DENSE_RANK)
top3_customers_per_city = spark.sql("""
    SELECT *
    FROM (
        SELECT
            customer_city,
            customer_id,
            customer_name,
            SUM(revenue)   AS total_revenue,
            DENSE_RANK() OVER (
                PARTITION BY customer_city
                ORDER BY SUM(revenue) DESC
            ) AS city_rank
        FROM car_sales
        GROUP BY customer_city, customer_id, customer_name
    ) ranked
    WHERE city_rank <= 3
    ORDER BY customer_city, city_rank
""")
print("=== Top 3 Customers per City ===")
top3_customers_per_city.show()

=== Top 3 Customers per City ===
+-------------+-----------+-------------+-------------+---------+
|customer_city|customer_id|customer_name|total_revenue|city_rank|
+-------------+-----------+-------------+-------------+---------+
|    Bangalore|         10|         Ravi|      50000.0|        1|
|    Bangalore|          2|        Rahul|      30000.0|        2|
|      Chennai|          6|        Kiran|      35000.0|        1|
|      Chennai|          9|        Divya|      22000.0|        2|
|      Chennai|          1|    Yashwanth|      20000.0|        3|
|        Delhi|          5|        Priya|      12000.0|        1|
|    Hyderabad|          3|        Sneha|      18000.0|        1|
|      Kolkata|          8|        Arjun|      17000.0|        1|
|       Mumbai|          4|         Amit|      24000.0|        1|
|         Pune|          7|        Meena|      80000.0|        1|
+-------------+-----------+-------------+-------------+---------+



In [0]:
# 6b. Monthly revenue trends
monthly_trends = spark.sql("""
    SELECT
        YEAR(sale_date)  AS sale_year,
        MONTH(sale_date) AS sale_month,
        COUNT(sale_id)   AS total_sales,
        SUM(revenue)     AS monthly_revenue
    FROM car_sales
    WHERE sale_date != '1970-01-01'          -- exclude sentinel (originally NULL) dates
    GROUP BY YEAR(sale_date), MONTH(sale_date)
    ORDER BY sale_year, sale_month
""")
print("=== Monthly Revenue Trends ===")
monthly_trends.show()

=== Monthly Revenue Trends ===
+---------+----------+-----------+---------------+
|sale_year|sale_month|total_sales|monthly_revenue|
+---------+----------+-----------+---------------+
|     2024|         1|          6|       139000.0|
|     2024|         2|          4|       169000.0|
+---------+----------+-----------+---------------+



In [0]:
# 6c. Repeat customers (customers who bought more than once)
repeat_customers = spark.sql("""
    SELECT
        customer_id,
        customer_name,
        COUNT(sale_id) AS purchase_count,
        SUM(revenue)   AS total_spent
    FROM car_sales
    GROUP BY customer_id, customer_name
    HAVING COUNT(sale_id) > 1
    ORDER BY purchase_count DESC
""")
print("=== Repeat Customers ===")
repeat_customers.show()

=== Repeat Customers ===
+-----------+-------------+--------------+-----------+
|customer_id|customer_name|purchase_count|total_spent|
+-----------+-------------+--------------+-----------+
+-----------+-------------+--------------+-----------+



# PHASE 7 – OUTPUT
# Save all final DataFrames to /tmp/car_sales_output (Parquet format).
# In Databricks Community Edition, /tmp is accessible via DBFS.

In [0]:
OUTPUT_BASE = "/Volumes/workspace/default/car_sales_output"

In [ ]:
customers_clean.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/customers_clean")
cars_clean.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/cars_clean")
dealers_clean.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/dealers_clean")
sales_clean.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/sales_clean")
sales_dealer_clean.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/sales_dealer_clean")

In [0]:
# Save transformation outputs
revenue_per_customer.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/revenue_per_customer")
brand_wise_sales.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/brand_wise_sales")
city_revenue.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/city_revenue")

In [0]:
# Save dealer analytics outputs
dealer_revenue.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/dealer_revenue")
dealer_city_contribution.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/dealer_city_contribution")

In [0]:
# Save SQL analysis outputs
top3_customers_per_city.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/top3_customers_per_city")
monthly_trends.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/monthly_trends")
repeat_customers.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/repeat_customers")
 
print("✅ All outputs saved to:", OUTPUT_BASE)

In [0]:
# Verify outputs were written (list directory)
display(dbutils.fs.ls(OUTPUT_BASE))